# APEX MD Analysis: RMSD & RMSF

Protein (chain A) -- RNA (chain B) complex, 6 replicates.

## Configuration

In [1]:
from __future__ import annotations

import gc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mplplots.palettes import CUSTOM
from mplplots.utils import auto_ticks

from mdpp.analysis.metrics import RMSDResult, RMSFResult, compute_rmsd, compute_rmsf
from mdpp.core.trajectory import load_trajectories
from mdpp.plots.timeseries import plot_rmsd, plot_rmsf, plot_rmsf_average

plt.style.use("mplplots.styles.GraphPadPrism")
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=CUSTOM)

In [2]:
DATA_DIR = Path("~/data/research/apex/md").expanduser()
REPLICAS = [f"gromacs{i}" for i in (1, 2, 3, 4, 7, 8)]
N_REPLICAS = len(REPLICAS)

# --- Trajectory files ---

TOPOLOGY = "step5_production_complex_fit.pdb"
TRAJECTORY = "step5_production_complex_fit.xtc"

# --- Frame selection (range-style: start included, stop excluded) ---

START = 0
STOP = None
STRIDE = 1

## Loading & Metrics

Load trajectories and compute per-replica RMSD (backbone, protein, RNA) and RMSF (CA).

**Note:** Trajectories are pre-aligned (GROMACS `gmx trjconv -fit rot+trans`).

In [ ]:
rmsd_backbone: dict[str, RMSDResult] = {}
rmsd_protein: dict[str, RMSDResult] = {}
rmsd_rna: dict[str, RMSDResult] = {}
rmsf_ca: dict[str, RMSFResult] = {}

sim_dirs = [DATA_DIR / replica for replica in REPLICAS]

loaded = load_trajectories(
    [sim_dir / TRAJECTORY for sim_dir in sim_dirs],
    topology_paths=[sim_dir / TOPOLOGY for sim_dir in sim_dirs],
    start=START,
    stop=STOP,
    stride=STRIDE,
    atom_selection="protein or is_nucleic",
    max_workers=N_REPLICAS,
)

for replica, traj in zip(REPLICAS, loaded, strict=True):
    rmsd_backbone[replica] = compute_rmsd(traj, atom_selection="backbone")
    rmsd_protein[replica] = compute_rmsd(traj, atom_selection="protein and backbone")
    rmsd_rna[replica] = compute_rmsd(traj, atom_selection="is_nucleic")
    rmsf_ca[replica] = compute_rmsf(traj, atom_selection="name CA")
    print(f"{replica}: {traj.n_frames} frames, {traj.n_atoms} atoms")

del loaded
gc.collect()
print("\nAll metrics computed.")

## RMSD (Backbone)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
for replica, rmsd in rmsd_backbone.items():
    plot_rmsd(rmsd, ax=ax, label=replica, alpha=0.3, moving_average=1000)
ax.set_title("Backbone RMSD")
auto_ticks(ax)
fig.tight_layout()
plt.show()

In [ ]:
ncols = 4
nrows = -(-N_REPLICAS // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), dpi=120, squeeze=False)
for idx, (replica, rmsd) in enumerate(rmsd_backbone.items()):
    ax = axes[idx // ncols][idx % ncols]
    plot_rmsd(rmsd, ax=ax, alpha=0.3, moving_average=100, color=CUSTOM[idx % len(CUSTOM)])
    ax.set_title(f"Backbone {replica}")
    auto_ticks(ax)
for idx in range(N_REPLICAS, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)
fig.tight_layout()
plt.show()

## RMSD (Protein)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
for replica, rmsd in rmsd_protein.items():
    plot_rmsd(rmsd, ax=ax, label=replica, alpha=0.3, moving_average=100)
ax.set_title("Protein Backbone RMSD")
auto_ticks(ax)
fig.tight_layout()
plt.show()

In [ ]:
ncols = 4
nrows = -(-N_REPLICAS // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), dpi=120, squeeze=False)
for idx, (replica, rmsd) in enumerate(rmsd_protein.items()):
    ax = axes[idx // ncols][idx % ncols]
    plot_rmsd(rmsd, ax=ax, alpha=0.3, moving_average=100, color=CUSTOM[idx % len(CUSTOM)])
    ax.set_title(f"Protein {replica}")
    auto_ticks(ax)
for idx in range(N_REPLICAS, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)
fig.tight_layout()
plt.show()

## RMSD (RNA)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
for replica, rmsd in rmsd_rna.items():
    plot_rmsd(rmsd, ax=ax, label=replica, alpha=0.3, moving_average=100)
ax.set_title("RNA RMSD")
auto_ticks(ax)
fig.tight_layout()
plt.show()

In [ ]:
ncols = 4
nrows = -(-N_REPLICAS // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), dpi=120, squeeze=False)
for idx, (replica, rmsd) in enumerate(rmsd_rna.items()):
    ax = axes[idx // ncols][idx % ncols]
    plot_rmsd(rmsd, ax=ax, alpha=0.3, moving_average=100, color=CUSTOM[idx % len(CUSTOM)])
    ax.set_title(f"RNA {replica}")
    auto_ticks(ax)
for idx in range(N_REPLICAS, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)
fig.tight_layout()
plt.show()

## RMSF

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), dpi=120)
rmsf_results = list(rmsf_ca.values())
for replica, rmsf in rmsf_ca.items():
    plot_rmsf(rmsf, ax=ax, label=replica, alpha=0.3)
plot_rmsf_average(rmsf_results, ax=ax, label="average", color="crimson", sem_alpha=0.6)
ax.set_title("CA RMSF")
fig.tight_layout()
plt.show()

In [ ]:
ncols = 4
nrows = -(-N_REPLICAS // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), dpi=120, squeeze=False)
for idx, (replica, rmsf) in enumerate(rmsf_ca.items()):
    ax = axes[idx // ncols][idx % ncols]
    plot_rmsf(rmsf, ax=ax, color=CUSTOM[idx % len(CUSTOM)])
    ax.set_title(replica)
for idx in range(N_REPLICAS, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)
fig.tight_layout()
plt.show()

## Clustering

Compare multiple clustering methods on the backbone RMSD matrix:

- **GROMOS** -- greedy largest-cluster-first (cutoff-based)
- **DBSCAN** -- density-based, labels noise frames as -1
- **HDBSCAN** -- hierarchical density-based, no epsilon tuning
- **Hierarchical** -- agglomerative average-linkage (subsampled for large N)
- **K-Means / Mini-Batch** -- feature-based on PCA projections

In [ ]:
import mdtraj as md

from mdpp.analysis.clustering import DBSCAN, HDBSCAN, Gromos, compute_rmsd_matrix

CLUSTER_STRIDE = 1
CUTOFF_NM = 0.35

sim_dirs = [DATA_DIR / replica for replica in REPLICAS]

trajs = load_trajectories(
    [sim_dir / TRAJECTORY for sim_dir in sim_dirs],
    topology_paths=[sim_dir / TOPOLOGY for sim_dir in sim_dirs],
    start=START,
    stop=STOP,
    stride=CLUSTER_STRIDE,
    atom_selection="protein or is_nucleic",
    max_workers=N_REPLICAS,
)

concat = md.join(trajs)
n_frames_per_replica = [t.n_frames for t in trajs]
del trajs
gc.collect()

print(f"Concatenated: {concat.n_frames} frames, {concat.n_atoms} atoms")
print(f"Frames per replica: {dict(zip(REPLICAS, n_frames_per_replica))}")

In [4]:
rmsd_mat = compute_rmsd_matrix(concat, atom_selection="backbone", backend="torch")

In [ ]:
# --- Run GROMOS, DBSCAN, HDBSCAN on the full RMSD matrix ---

methods: dict[str, object] = {}

methods["GROMOS"] = Gromos(cutoff_nm=CUTOFF_NM)(rmsd_mat.rmsd_matrix_nm)

methods["DBSCAN"] = DBSCAN(eps=CUTOFF_NM, min_samples=5)(rmsd_mat.rmsd_matrix_nm)

methods["HDBSCAN"] = HDBSCAN(min_cluster_size=50, min_samples=5)(rmsd_mat.rmsd_matrix_nm)

# --- Summary table ---

n_total = concat.n_frames
print(f"{'Method':<12s} {'Clusters':>10s} {'Noise':>8s} {'Largest cluster':>18s}")
print("-" * 50)
for name, result in methods.items():
    noise = int(np.sum(result.labels == -1))
    if result.n_clusters > 0:
        largest = int(np.bincount(result.labels[result.labels >= 0]).max())
    else:
        largest = 0
    pct = largest / n_total * 100
    print(f"{name:<12s} {result.n_clusters:>10d} {noise:>8d} {largest:>10d} ({pct:5.1f}%)")

In [ ]:
# --- Cluster populations: top-20 per method ---

fig, axes = plt.subplots(1, 3, figsize=(18, 4), dpi=120, sharey=True)

for ax, (name, result) in zip(axes, methods.items(), strict=False):
    valid = result.labels[result.labels >= 0]
    if len(valid) > 0:
        counts = np.bincount(valid)
        top_k = min(20, len(counts))
        ax.bar(range(top_k), counts[:top_k], color=CUSTOM[0])
    ax.set_xlabel("Cluster")
    ax.set_title(f"{name} ({result.n_clusters} clusters)")
    auto_ticks(ax)

axes[0].set_ylabel("Frames")
fig.suptitle(f"Cluster Populations (cutoff = {CUTOFF_NM} nm)", y=1.02)
fig.tight_layout()
plt.show()

### Hierarchical Clustering (subsampled)

Scipy's agglomerative linkage requires an O(N^2) condensed distance matrix
in float64, which is prohibitive at 120k frames. Subsample first.

In [ ]:
from mdpp.analysis.clustering import Hierarchical

HIER_SUBSAMPLE = 10
sub_matrix = rmsd_mat.rmsd_matrix_nm[::HIER_SUBSAMPLE, ::HIER_SUBSAMPLE]
n_sub = sub_matrix.shape[0]
print(f"Subsampled: {n_sub} frames (every {HIER_SUBSAMPLE}th)")

hier_results: dict[str, object] = {}
for linkage in ("average", "complete", "single"):
    hier_results[linkage] = Hierarchical(
        linkage_method=linkage,
        distance_threshold=CUTOFF_NM,
    )(sub_matrix)
    print(f"  {linkage}: {hier_results[linkage].n_clusters} clusters")

# --- Plot ---

fig, axes = plt.subplots(1, 3, figsize=(18, 4), dpi=120, sharey=True)
for ax, (linkage, result) in zip(axes, hier_results.items(), strict=False):
    counts = np.bincount(result.labels)
    top_k = min(20, len(counts))
    ax.bar(range(top_k), counts[:top_k], color=CUSTOM[0])
    ax.set_xlabel("Cluster")
    ax.set_title(f"Hierarchical {linkage} ({result.n_clusters} clusters)")
    auto_ticks(ax)

axes[0].set_ylabel("Frames")
fig.suptitle(f"Hierarchical Linkage Comparison (subsampled {HIER_SUBSAMPLE}x)", y=1.02)
fig.tight_layout()
plt.show()

### Feature-Based Clustering (PCA + K-Means)

Backbone torsion featurization (sin/cos embedded phi/psi) followed by PCA.
K-Means and Mini-Batch K-Means cluster in the reduced feature space --
scales linearly with N and does not require the full RMSD matrix.

In [ ]:
from mdpp.analysis.clustering import KMeans, MiniBatchKMeans
from mdpp.analysis.decomposition import compute_pca, featurize_backbone_torsions

# --- Featurize backbone torsions ---

torsions = featurize_backbone_torsions(concat, atom_selection="protein")
pca = compute_pca(torsions.values, n_components=10)

print(f"Torsion features: {torsions.values.shape[1]}")
print(
    f"PCA: {pca.projections.shape[1]} components, "
    f"explained variance = {pca.explained_variance_ratio.sum():.1%}"
)

# --- K-Means and Mini-Batch K-Means ---

N_CLUSTERS_PCA = 10

km = KMeans(n_clusters=N_CLUSTERS_PCA)(pca.projections)
mb = MiniBatchKMeans(n_clusters=N_CLUSTERS_PCA)(pca.projections)

print(f"\nK-Means:     {km.n_clusters} clusters, inertia = {km.inertia:.1f}")
print(f"Mini-Batch:  {mb.n_clusters} clusters, inertia = {mb.inertia:.1f}")

In [ ]:
# --- PC1 vs PC2 scatter, colored by cluster labels ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

for ax, (name, result) in zip(axes, [("K-Means", km), ("Mini-Batch", mb)], strict=False):
    sc = ax.scatter(
        pca.projections[:, 0],
        pca.projections[:, 1],
        c=result.labels,
        cmap="tab10",
        s=1,
        alpha=0.3,
        rasterized=True,
    )
    # Mark cluster centers
    ax.scatter(
        result.cluster_centers[:, 0],
        result.cluster_centers[:, 1],
        c="black",
        marker="x",
        s=80,
        linewidths=2,
        zorder=5,
    )
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"{name} ({result.n_clusters} clusters)")

fig.tight_layout()
plt.show()

In [ ]:
# --- RMSD matrix heatmap ---

fig, ax = plt.subplots(figsize=(7, 6), dpi=120)
im = ax.imshow(rmsd_mat.rmsd_matrix_angstrom, cmap="viridis", origin="lower", aspect="equal")
fig.colorbar(im, ax=ax, label="RMSD (A)")
ax.set_xlabel("Frame")
ax.set_ylabel("Frame")
ax.set_title("Pairwise RMSD Matrix")
fig.tight_layout()
plt.show()

In [ ]:
# --- Save GROMOS medoid structures ---

gromos = methods["GROMOS"]
for i, frame_idx in enumerate(gromos.medoid_frames):
    out = DATA_DIR / f"cluster{i}_medoid.pdb"
    concat[int(frame_idx)].save(str(out))
    print(f"Cluster {i} medoid -> {out}")

del concat, rmsd_mat, torsions, pca
gc.collect()